# **TaniMol: 07 - Export Results**

This notebook exports all computed results from the pipeline into portable, reusable files. The goal is to produce a self-contained `results/` directory that can be shared independently of the codebase — anyone receiving it gets the full picture without needing to rerun the pipeline.

**Exported files:**
- `cluster_statistics.csv` — per-cluster pIC50 summary (mean, median, std, min, max)
- `activity_cliffs.csv` — all detected cliff pairs with SMILES, similarity, ΔpIC50, SALI
- `molecule_summary.csv` — per-molecule table with cluster assignment, max SALI, cliff involvement
- `pipeline_summary.json` — run metadata, parameters, and key results
- `figures/` — all plots as PNG (300 DPI, print quality)

### **1. Load Data and Recompute Results**

Load the clustering results and preprocessed dataset, then recompute all activity analysis outputs. This ensures the exports reflect the exact same state as the analysis notebooks.

In [ ]:
from src.config import PROCESSED_DIR
from src.activity_analysis import (
    within_cluster_activity_distributions, activity_cliffs, sali,
    similarity_activity_correlation,
)
from src.export import (
    export_cluster_statistics, export_activity_cliffs,
    export_molecule_summary, export_pipeline_summary, export_all_figures,
)
import pandas as pd
import numpy as np
import pickle

with open(f"{PROCESSED_DIR}/clustering_results.pkl", "rb") as f:
    results = pickle.load(f)

morgan_sim = results["morgan"]["similarity_matrix"]
morgan_clusters = results["morgan"]["clusters"]
morgan_singletons = results["morgan"]["singletons"]

df = pd.read_csv(f"{PROCESSED_DIR}/cleaned_activities.csv")
pic50 = df["pchembl_value"].values
delta_pic50 = np.abs(pic50[:, None] - pic50[None, :])

cluster_stats = within_cluster_activity_distributions(morgan_clusters, pic50)
cliffs = activity_cliffs(morgan_sim, delta_pic50)
sali_matrix, sali_values = sali(morgan_sim, delta_pic50)
rho, p_value = similarity_activity_correlation(morgan_sim, delta_pic50)

### **2. Export Data**

Write all tabular results and metadata to `results/`. Each file is self-contained and can be opened in Excel, pandas, or any JSON viewer without additional context.

In [ ]:
export_cluster_statistics(cluster_stats)
export_activity_cliffs(cliffs, df, sali_matrix)
export_molecule_summary(df, morgan_clusters, morgan_singletons, sali_matrix, cliffs)
export_pipeline_summary(df, morgan_clusters, morgan_singletons, cliffs, sali_values, rho, p_value)

### **3. Export Figures**

Save all activity analysis plots to `results/figures/` as high-resolution PNGs. These are publication-ready — white background, 300 DPI, tight bounding boxes.

In [ ]:
from src.visualization import (
    plot_cluster_activity_boxplots, plot_activity_cliff_scatter,
    plot_sali_distribution, plot_similarity_activity_density,
)

export_all_figures({
    "cluster_activity_boxplots": lambda: plot_cluster_activity_boxplots(morgan_clusters, pic50),
    "activity_cliff_scatter": lambda: plot_activity_cliff_scatter(morgan_sim, delta_pic50),
    "sali_distribution": lambda: plot_sali_distribution(sali_values),
    "similarity_activity_density": lambda: plot_similarity_activity_density(morgan_sim, delta_pic50, rho, p_value),
})